# Bias Detection, Fairness Evaluation, and Mitigation
This notebook is the interactive version of the full research pipeline.


## 0. Install Dependencies


In [ ]:
!pip install pandas numpy scikit-learn torch fairlearn shap scipy ucimlrepo


## 1. Data Fetching


In [ ]:
"""
Script to dynamically fetch and cache datasets from the UCI ML Repository.
Focuses on Adult Income (id=2) and Heart Disease (id=45).
"""
import pandas as pd
from ucimlrepo import fetch_ucirepo
import os

DATA_DIR = os.path.join(os.path.dirname(__file__), 'raw_data')
os.makedirs(DATA_DIR, exist_ok=True)

def fetch_adult_income():
    print("Fetching Adult Income dataset...")
    try:
        adult = fetch_ucirepo(id=2) 
        X = adult.data.features 
        y = adult.data.targets 
        df = pd.concat([X, y], axis=1)
        path = os.path.join(DATA_DIR, 'adult.csv')
        df.to_csv(path, index=False)
        print(f"Adult Income saved to {path}")
        return df
    except Exception as e:
        print(f"Error fetching Adult Income: {e}")
        return None

def fetch_heart_disease():
    print("Fetching Heart Disease dataset...")
    try:
        heart_disease = fetch_ucirepo(id=45) 
        X = heart_disease.data.features 
        y = heart_disease.data.targets 
        df = pd.concat([X, y], axis=1)
        path = os.path.join(DATA_DIR, 'heart_disease.csv')
        df.to_csv(path, index=False)
        print(f"Heart Disease saved to {path}")
        return df
    except Exception as e:
        print(f"Error fetching Heart Disease: {e}")
        return None

if __name__ == "__main__":
    df_adult = fetch_adult_income()
    df_heart = fetch_heart_disease()



In [ ]:
df_adult = fetch_adult_income()
df_adult.head()


## 2. Preprocessing & Bias Symptoms


In [ ]:
"""
01_preprocessing_and_symptoms.py
Handles Data Preprocessing and initial Bias Detection using dataset symptoms.
Provides dataset diagnostic functionality.
"""
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import mutual_info_classif
from scipy.stats import wasserstein_distance

DATA_DIR = os.path.join(os.path.dirname(os.path.dirname(__file__)), 'datasets', 'raw_data')

def load_data(dataset_name='adult.csv'):
    path = os.path.join(DATA_DIR, dataset_name)
    if not os.path.exists(path):
        print(f"Data not found at {path}. Please run datasets/fetch_data.py first.")
        return None
    
    df = pd.read_csv(path)
    # Basic cleaning
    df.replace('?', np.nan, inplace=True)
    df.dropna(inplace=True)
    return df

def preprocess_data(df, target_col='income', protected_attr='sex'):
    print(f"--- Preprocessing Data ---")
    le = LabelEncoder()
    # Encode target
    df[target_col] = le.fit_transform(df[target_col].astype(str))
    
    # Encode categorical features
    categorical_cols = df.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        df[col] = le.fit_transform(df[col].astype(str))
    
    # Numerical normalization
    num_cols = df.select_dtypes(include=['int64', 'float64']).columns
    num_cols = [c for c in num_cols if c not in [target_col, protected_attr] and c in df.columns]
    
    if len(num_cols) > 0:
        scaler = StandardScaler()
        df[num_cols] = scaler.fit_transform(df[num_cols])
        
    print(f"Preprocessed DataFrame shape: {df.shape}")
    return df

def calculate_bias_symptoms(df, target_col='income', protected_attr='sex'):
    """
    Quantifies intrinsic dataset bias before model training.
    Uses Mutual Information and Earth Mover's Distance (Wasserstein Distance).
    """
    print(f"\n--- Calculating Bias Symptoms ---")
    
    # 1. Mutual Information between Protected Attribute and Target
    if df[protected_attr].dtype == 'float64':
        df[protected_attr] = df[protected_attr].astype(int)
    
    mi = mutual_info_classif(df[[protected_attr]], df[target_col], discrete_features=True)
    print(f"Mutual Information ({protected_attr} -> {target_col}): {mi[0]:.4f}")
    
    # 2. Earth Mover's Distance (EMD / Wasserstein Distance) 
    # Compare the distribution of the target based on the protected attribute groups
    groups = df[protected_attr].unique()
    if len(groups) >= 2:
        group_0_target = df[df[protected_attr] == groups[0]][target_col].values
        group_1_target = df[df[protected_attr] == groups[1]][target_col].values
        
        emd = wasserstein_distance(group_0_target, group_1_target)
        print(f"Earth Mover's Distance (EMD) between group {groups[0]} and {groups[1]} for {target_col}: {emd:.4f}")
    
    return mi[0], emd if len(groups) >= 2 else 0

if False:
    df = load_data()
    if df is not None:
        df_processed = preprocess_data(df, target_col='income', protected_attr='sex')
        calculate_bias_symptoms(df_processed, target_col='income', protected_attr='sex')
        
        # Save processed data explicitly for the user's research
        processed_dir = os.path.join(os.path.dirname(DATA_DIR), 'processed_data')
        os.makedirs(processed_dir, exist_ok=True)
        out_path = os.path.join(processed_dir, 'adult_processed.csv')
        df_processed.to_csv(out_path, index=False)
        print(f"Processed data saved securely to: {out_path}")




In [ ]:
df_processed = preprocess_data(df_adult, target_col='income', protected_attr='sex')
calculate_bias_symptoms(df_processed, target_col='income', protected_attr='sex')


## 3. Baseline Training


In [ ]:
"""
02_baseline_training.py
Trains standard (Unfair) models.
"""
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
import joblib

DATA_DIR = os.path.join(os.path.dirname(os.path.dirname(__file__)), 'datasets', 'raw_data')
MODEL_DIR = os.path.join(os.path.dirname(os.path.dirname(__file__)), 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

def train_baseline(df, target_col='income', protected_attr='sex'):
    print("\n--- Training Baseline Models ---")
    
    # Separate features and target
    X = df.drop(columns=[target_col])
    y = df[target_col]
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    print(f"Training shape: {X_train.shape}, Testing shape: {X_test.shape}")
    
    models = {
        'LogisticRegression': LogisticRegression(max_iter=1000),
        'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42)
    }
    
    results = {}
    for name, model in models.items():
        print(f"\nTraining {name}...")
        model.fit(X_train, y_train)
        
        preds = model.predict(X_test)
        preds_proba = model.predict_proba(X_test)[:, 1]
        
        acc = accuracy_score(y_test, preds)
        auc = roc_auc_score(y_test, preds_proba)
        
        print(f"{name} Results:")
        print(f"Accuracy: {acc:.4f} | ROC AUC: {auc:.4f}")
        
        # Save model
        joblib.dump(model, os.path.join(MODEL_DIR, f"baseline_{name}.pkl"))
        
        results[name] = {
            'accuracy': acc,
            'auc': auc,
            'model': model
        }
        
    # We return the test sets to be used by the fairness evaluation
    return results, X_test, y_test

if False:
    from importlib.machinery import SourceFileLoader
    preproc = SourceFileLoader("preproc", os.path.join(os.path.dirname(__file__), "01_preprocessing_and_symptoms.py")).load_module()
    
    df = preproc.load_data()
    if df is not None:
        df_processed = preproc.preprocess_data(df)
        train_baseline(df_processed)



In [ ]:
results, X_test, y_test = train_baseline(df_processed)


## 4. Fairness Evaluation


In [ ]:
"""
03_fairness_evaluation.py
Implements Multi-Metric Fairness Evaluation.
Calculates Demographic Parity, Equalized Odds, Disparate Impact natively and using fairlearn.
Includes concepts like Propensity Score Matching for Matched Counterpart Auditing.
"""
import os
import joblib
import pandas as pd
import numpy as np
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference
from sklearn.neighbors import NearestNeighbors

MODEL_DIR = os.path.join(os.path.dirname(os.path.dirname(__file__)), 'models')

def calculate_disparate_impact(y_true, y_pred, sensitive_features, group_1, group_0):
    """
    Calculates Disparate Impact (DI). 
    DI = P(y_pred=1 | group=group_1) / P(y_pred=1 | group=group_0)
    """
    df = pd.DataFrame({'y_pred': y_pred, 'group': sensitive_features})
    prob_1 = df[df['group'] == group_1]['y_pred'].mean()
    prob_0 = df[df['group'] == group_0]['y_pred'].mean()
    
    if prob_0 == 0:
        return np.inf
    return prob_1 / prob_0

def evaluate_model_fairness(model, X_test, y_test, protected_attr='sex', group_1=1, group_0=0):
    print(f"\n--- Fairness Evaluation for Model ---")
    sensitive_features = X_test[protected_attr]
    y_pred = model.predict(X_test)
    
    dp_diff = demographic_parity_difference(y_test, y_pred, sensitive_features=sensitive_features)
    eo_diff = equalized_odds_difference(y_test, y_pred, sensitive_features=sensitive_features)
    di = calculate_disparate_impact(y_test, y_pred, sensitive_features, group_1, group_0)
    
    print(f"Demographic Parity Difference: {dp_diff:.4f} (Ideal: 0)")
    print(f"Equalized Odds Difference:     {eo_diff:.4f} (Ideal: 0)")
    print(f"Disparate Impact:              {di:.4f} (Ideal: 1)")
    
    return {'dp_diff': dp_diff, 'eo_diff': eo_diff, 'di': di}

def matched_counterpart_auditing(X_test, y_pred, protected_attr='sex', group_1=1, group_0=0):
    """
    Simulates Propensity Score Matching / Matched Counterparts auditing from Objective 2.
    We match individuals from group_1 to individuals in group_0 based on their covariates (excluding the protected attr).
    """
    print("\n--- Auditing with Matched Counterparts ---")
    X_g1 = X_test[X_test[protected_attr] == group_1].drop(columns=[protected_attr])
    X_g0 = X_test[X_test[protected_attr] == group_0].drop(columns=[protected_attr])
    
    y_pred_g1 = y_pred[X_test[protected_attr] == group_1]
    y_pred_g0 = y_pred[X_test[protected_attr] == group_0]
    
    # Simple K-NN matching (k=1)
    if len(X_g1) > 0 and len(X_g0) > 0:
        nn = NearestNeighbors(n_neighbors=1).fit(X_g0)
        distances, indices = nn.kneighbors(X_g1)
        
        # Compare outcomes of matched pairs
        matched_g0_preds = y_pred_g0.iloc[indices.flatten()].values
        matching_discrepancy = np.mean(y_pred_g1 != matched_g0_preds)
        
        print(f"Matched Counterpart Discrepancy Rate: {matching_discrepancy:.4f}")
        print(f"Measures systematic bias ignoring covariate differences.")
        return matching_discrepancy
    return None

if False:
    from importlib.machinery import SourceFileLoader
    preproc = SourceFileLoader("preproc", os.path.join(os.path.dirname(__file__), "01_preprocessing_and_symptoms.py")).load_module()
    baseline = SourceFileLoader("baseline", os.path.join(os.path.dirname(__file__), "02_baseline_training.py")).load_module()
    
    df = preproc.load_data()
    if df is not None:
        df_processed = preproc.preprocess_data(df)
        _, X_test, y_test = baseline.train_baseline(df_processed)
        
        # Load the baseline random forest to evaluate
        model_path = os.path.join(MODEL_DIR, "baseline_RandomForest.pkl")
        if os.path.exists(model_path):
            rf_model = joblib.load(model_path)
            evaluate_model_fairness(rf_model, X_test, y_test)
            y_pred = pd.Series(rf_model.predict(X_test), index=X_test.index)
            matched_counterpart_auditing(X_test, y_pred)



In [ ]:
evaluate_model_fairness(results['RandomForest']['model'], X_test, y_test)
y_pred = pd.Series(results['RandomForest']['model'].predict(X_test), index=X_test.index)
matched_counterpart_auditing(X_test, y_pred)


## 5. Advanced Mitigation (Fair NN)


In [ ]:
"""
04_mitigation.py
Implements advanced mitigation strategies.
1. PyTorch Neural Network with Differentiable Fairness Penalty (Proxy for MI Regularizer)
2. BiasPruner conceptual approach (pruning weights most correlated with protected attribute)
"""
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

MODEL_DIR = os.path.join(os.path.dirname(os.path.dirname(__file__)), 'models')

class FairNeuralNet(nn.Module):
    def __init__(self, input_dim):
        super(FairNeuralNet, self).__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        h1 = self.relu(self.fc1(x))
        h2 = self.relu(self.fc2(h1))
        out = self.sigmoid(self.fc3(h2))
        return out, h2 # Returning hidden layer for BiasPruner

def demographic_parity_loss(outputs, sensitive_attributes):
    """
    Differentiable proxy for Demographic Parity to use as a regularizer.
    Penalizes the squared difference between the mean prediction for group 0 and group 1.
    """
    mask_g1 = (sensitive_attributes == 1.0)
    mask_g0 = (sensitive_attributes == 0.0)
    
    if mask_g1.sum() == 0 or mask_g0.sum() == 0:
        return torch.tensor(0.0, device=outputs.device)
        
    mean_g1 = torch.mean(outputs[mask_g1])
    mean_g0 = torch.mean(outputs[mask_g0])
    
    return torch.pow(mean_g1 - mean_g0, 2)

def train_fair_model(X, y, protected_attr_idx, lambda_fairness=0.5, epochs=50):
    print("\n--- Training Fair Neural Network (Differentiable Regularizer) ---")
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    
    # Convert to PyTorch tensors
    X_train_t = torch.FloatTensor(X_train.values)
    y_train_t = torch.FloatTensor(y_train.values).unsqueeze(1)
    sensitive_train = X_train_t[:, protected_attr_idx]
    
    X_test_t = torch.FloatTensor(X_test.values)
    
    model = FairNeuralNet(X_train.shape[1])
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    bce_loss = nn.BCELoss()
    
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs, _ = model(X_train_t)
        
        # Standard Objective
        loss_main = bce_loss(outputs, y_train_t)
        
        # Fairness Objective (Mutual Information / DP proxy)
        loss_fair = demographic_parity_loss(outputs, sensitive_train)
        
        # Total Loss
        total_loss = loss_main + lambda_fairness * loss_fair
        total_loss.backward()
        optimizer.step()
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} | Main Loss: {loss_main.item():.4f} | Fair Loss: {loss_fair.item():.4f}")
            
    # Evaluation
    model.eval()
    with torch.no_grad():
        test_outputs, _ = model(X_test_t)
        preds = (test_outputs.squeeze() > 0.5).numpy().astype(int)
        acc = accuracy_score(y_test, preds)
        print(f"Fair NN Accuracy: {acc:.4f}")
        
    torch.save(model.state_dict(), os.path.join(MODEL_DIR, "fair_nn.pth"))
    print("Fair Model saved successfully.")
    
    return model, X_test, y_test, preds

if False:
    from importlib.machinery import SourceFileLoader
    preproc = SourceFileLoader("preproc", os.path.join(os.path.dirname(__file__), "01_preprocessing_and_symptoms.py")).load_module()
    
    df = preproc.load_data()
    if df is not None:
        df_processed = preproc.preprocess_data(df)
        X = df_processed.drop(columns=['income'])
        y = df_processed['income']
        # Find index of 'sex'
        protected_attr_idx = X.columns.get_loc('sex')
        train_fair_model(X, y, protected_attr_idx)



In [ ]:
X = df_processed.drop(columns=['income'])
y = df_processed['income']
protected_attr_idx = X.columns.get_loc('sex')
fair_model, _, _, _ = train_fair_model(X, y, protected_attr_idx)
